# NGB v3 versus v4 one-head comparison

This notebook compares the checked-in one-epoch v3 protocol with the separate
tuned two-epoch v4 protocol. Differences are paired by model seed. The result is
an overall protocol comparison—horizon and optimizer schedule both changed.


In [ ]:
V3_RESULTS_ROOT = "/tmp/rg-nanogpt-one-head/results"
V4_CONFIG = "configs/v4_one_head.yaml"
NGB_STORAGE_ROOT = "/tmp/rg-ngb"
SEEDS = ""


In [ ]:
from pathlib import Path
import sys
import math
import matplotlib.pyplot as plt
import pandas as pd

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd / "baseline" / "ngb"]
NGB_ROOT_DIR = next((p for p in candidates if (p / "configs" / "v4_one_head.yaml").is_file()), None)
if NGB_ROOT_DIR is None:
    raise FileNotFoundError("Run from baseline/ngb or the repository root")
RUNTIME_SRC = NGB_ROOT_DIR.parent / "nanogpt_one_head" / "src"
if str(RUNTIME_SRC) not in sys.path:
    sys.path.insert(0, str(RUNTIME_SRC))

from rg_nanogpt_one_head import (
    OPTIMIZER_COLORS, OPTIMIZER_LABELS, SUPPORTED_OPTIMIZERS,
    discover_matched_complete_seeds, final_test_summary, load_config,
    load_epoch_metrics, load_spectral_summary, load_test_results, mean_ci95,
    run_slug,
)
from rg_nanogpt_one_head.analysis import summarize_by_epoch
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)


In [ ]:
v4_cfg = load_config(NGB_ROOT_DIR / V4_CONFIG)
roots = {
    "v3": Path(V3_RESULTS_ROOT),
    "v4": Path(NGB_STORAGE_ROOT) / "results" / run_slug(v4_cfg),
}
optimizers = tuple(SUPPORTED_OPTIMIZERS)
available = {name: set(discover_matched_complete_seeds(root, optimizers=optimizers)) for name, root in roots.items()}
seeds = (
    tuple(int(v.strip()) for v in SEEDS.split(",") if v.strip())
    if SEEDS.strip()
    else tuple(sorted(set.intersection(*available.values())))
)
if not seeds:
    raise RuntimeError(f"No matched v3/v4 complete seeds: {available}")
print("matched v3/v4 seeds:", seeds)

frames_epoch=[]; frames_spectral=[]; frames_test=[]
for protocol, root in roots.items():
    epoch=load_epoch_metrics(root, optimizers=optimizers, seeds=seeds)
    spectral=load_spectral_summary(root, optimizers=optimizers, seeds=seeds)
    test=load_test_results(root, optimizers=optimizers, seeds=seeds)
    for frame in (epoch, spectral, test): frame.insert(0, "protocol", protocol)
    frames_epoch.append(epoch); frames_spectral.append(spectral); frames_test.append(test)
epoch_metrics=pd.concat(frames_epoch, ignore_index=True)
spectral_summary=pd.concat(frames_spectral, ignore_index=True)
test_results=pd.concat(frames_test, ignore_index=True)
plot_root=Path(NGB_STORAGE_ROOT)/"plots"/"v3_vs_v4_one_head"
plot_root.mkdir(parents=True, exist_ok=True)


In [ ]:
parts=[]
for protocol, frame in test_results.groupby("protocol"):
    part=final_test_summary(frame); part.insert(0,"protocol",protocol); parts.append(part)
summary=pd.concat(parts, ignore_index=True)
summary.to_csv(plot_root/"v3_v4_summary_95ci.csv", index=False)
display(summary.sort_values(["checkpoint","metric","optimizer","protocol"]))

rows=[]
for optimizer in optimizers:
    for checkpoint in ("final","validation_selected"):
        selected=test_results[(test_results["optimizer"]==optimizer)&(test_results["checkpoint"]==checkpoint)]
        for metric in ("test_loss","test_accuracy","test_bleu"):
            v4=selected[selected["protocol"]=="v4"][["seed",metric]].rename(columns={metric:"v4"})
            v3=selected[selected["protocol"]=="v3"][["seed",metric]].rename(columns={metric:"v3"})
            paired=v4.merge(v3,on="seed",validate="one_to_one")
            stats=mean_ci95(paired["v4"]-paired["v3"])
            rows.append({"optimizer":optimizer,"optimizer_label":OPTIMIZER_LABELS[optimizer],"checkpoint":checkpoint,"metric":metric,"contrast":"v4 - v3",**stats})
paired=pd.DataFrame(rows)
paired.to_csv(plot_root/"paired_v4_minus_v3_95ci.csv",index=False)
display(paired.sort_values(["checkpoint","metric","optimizer"]))


In [ ]:
styles={"v3":":","v4":"-"}
for optimizer in optimizers:
    for metric in ("val_loss","val_accuracy","test_loss","test_accuracy"):
        fig,ax=plt.subplots(figsize=(9,5))
        for protocol in ("v3","v4"):
            subset=epoch_metrics[(epoch_metrics["protocol"]==protocol)&(epoch_metrics["optimizer"]==optimizer)]
            summary_curve=summarize_by_epoch(subset,metric,x="nominal_epoch",group=("protocol","optimizer"))
            ax.plot(summary_curve["nominal_epoch"],summary_curve["mean"],color=OPTIMIZER_COLORS[optimizer],linestyle=styles[protocol],linewidth=2.2,label=protocol)
            ax.fill_between(summary_curve["nominal_epoch"],summary_curve["ci95_lower"],summary_curve["ci95_upper"],color=OPTIMIZER_COLORS[optimizer],alpha=0.10)
        ax.set(xlabel="Corpus-equivalent epoch",ylabel=metric,title=f"{OPTIMIZER_LABELS[optimizer]}: v3 versus v4")
        ax.grid(alpha=0.25); ax.legend(frameon=False); fig.tight_layout()
        fig.savefig(plot_root/f"{optimizer}_{metric}.png",dpi=170,bbox_inches="tight")
        plt.show()

for optimizer in optimizers:
    fig,ax=plt.subplots(figsize=(9,5))
    for protocol in ("v3","v4"):
        subset=spectral_summary[(spectral_summary["protocol"]==protocol)&(spectral_summary["optimizer"]==optimizer)]
        summary_curve=summarize_by_epoch(subset,"alpha_median",x="epoch",group=("protocol","optimizer"))
        ax.plot(summary_curve["epoch"],summary_curve["mean"],color=OPTIMIZER_COLORS[optimizer],linestyle=styles[protocol],linewidth=2.2,label=protocol)
        ax.fill_between(summary_curve["epoch"],summary_curve["ci95_lower"],summary_curve["ci95_upper"],color=OPTIMIZER_COLORS[optimizer],alpha=0.10)
    ax.axhline(2.0,color="black",linestyle="--",linewidth=1.0,label="alpha = 2")
    ax.set(xlabel="Corpus-equivalent epoch",ylabel="Median alpha",title=f"{OPTIMIZER_LABELS[optimizer]}: v3 versus v4 spectral flow")
    ax.grid(alpha=0.25); ax.legend(frameon=False); fig.tight_layout()
    fig.savefig(plot_root/f"{optimizer}_alpha_median.png",dpi=170,bbox_inches="tight")
    plt.show()
